# 06 · Built-in tools

**Goal:** use three tools that Anthropic designed: a **text editor** that Claude uses on your files, **web search**, and **code execution** with the **Files API**.

| | |
|---|---|
| **You will learn** | The three kinds of tools, the text editor tool and how to keep it safe, web search with citations, and running code in Anthropic's sandbox on a file you upload |
| **Prerequisites** | [Notebook 05](05_tool_use.ipynb), for the tool loop |
| **Run time** | About 3 minutes |
| **Cost** | Under $0.15 expected. This notebook stops itself at $0.30 |
| **Paid cells** | Sections 2c, 2d, 2e, 3 and 4. The rest is free |

If you haven't yet, set a spend limit in the [Claude Console](https://platform.claude.com/settings/billing) (**Billing**, then **Spend limits**).

## Setup

The same setup as the earlier notebooks, plus the conversation helpers from notebook 01. Run the cells from top to bottom.

`track()` has one addition here: web search is billed per search, at $10 for every 1,000 searches, on top of tokens. The response reports the number of searches in `usage.server_tool_use`, so `track()` adds that to the cost.

In [ ]:
import io
import json
import os
import subprocess
import sys
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv

load_dotenv(override=True)  # reads your keys from the .env file

api_key = os.getenv("ANTHROPIC_API_KEY", "")
assert api_key and api_key != "your_key_here", "Add your API key to .env, restart the kernel, and run this again."

client = Anthropic()  # finds ANTHROPIC_API_KEY by itself
MODEL = "claude-sonnet-5-5"

OUTPUT_DIR = Path("outputs/06_built_in_tools")  # everything this notebook creates goes here
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Ready. Using model:", MODEL)

In [ ]:
INPUT_PRICE = 2.00              # dollars per 1 million input tokens (Sonnet 5.5)
OUTPUT_PRICE = 10.00            # dollars per 1 million output tokens (Sonnet 5.5)
WEB_SEARCH_PRICE = 10 / 1000    # dollars per web search
BUDGET_USD = 0.30               # the most this notebook should spend

spent = 0.0


def track(response):
    global spent
    input_tokens = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    server_tools = response.usage.server_tool_use
    searches = server_tools.web_search_requests if server_tools else 0
    cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000 + searches * WEB_SEARCH_PRICE
    spent += cost
    print(f"[input={input_tokens:,} output={output_tokens:,} searches={searches} "
          f"cost=${cost:.5f} | spent ${spent:.4f} of ${BUDGET_USD:.2f}]")
    if spent > BUDGET_USD:
        raise RuntimeError(f"Budget reached: spent ${spent:.4f}. Raise BUDGET_USD if you mean to spend more.")
    return response


def text_from_message(message):
    for block in message.content:
        if block.type == "text":
            return block.text
    return ""


def full_text(blocks):
    """Join ALL text blocks. Answers with citations arrive split across many text blocks."""
    return "".join(block.text for block in blocks if block.type == "text")


def add_user_message(messages, content):
    messages.append({"role": "user", "content": content})


def add_assistant_message(messages, response):
    messages.append({"role": "assistant", "content": response.content})

## 1. Three kinds of tools (free)

In notebook 05 you wrote every tool yourself. Anthropic also provides tools of its own, and they come in two kinds:

| Kind | Who describes the tool | Who runs it | Do you need the tool loop? | Example |
|---|---|---|---|---|
| **Your own tools** | You write the schema | You | Yes | `set_reminder`, notebook 05 |
| **Anthropic-defined, you run it** | Built into Claude: you send a short stub | You | Yes | Text editor, section 2 |
| **Server tools** | Built into Claude: you send a short stub | **Anthropic's servers** | **No** | Web search and code execution, sections 3 and 4 |

A **stub** is just a `type` and a `name`, with no description and no `input_schema`. Claude already knows how the tool works, because it was trained on it.

## 2. The text editor tool

Claude can view, create and edit files with this tool. Claude sends the commands, and **your code touches the disk**:

| `command` | Inputs | What your code must do |
|---|---|---|
| `view` | `path`, and optionally `view_range` | Return the file with line numbers, or list a folder |
| `create` | `path`, `file_text` | Create the file, or overwrite it |
| `str_replace` | `path`, `old_str`, `new_str` | Replace text that appears **exactly once**. Return an error for zero or several matches |
| `insert` | `path`, `insert_line`, `insert_text` | Insert text after a line. Line `0` means the top of the file |

> **Different from the course:** the course's stub, `text_editor_20250124` named `str_replace_editor`, is for Claude 3.7. Claude 4 and later models, including Sonnet 5.5, use **`text_editor_20250728`** named **`str_replace_based_edit_tool`**. The course's `undo_edit` command no longer exists.

**Safety first.** The `path` comes from the model, so treat it as untrusted. Every file operation here is confined to one **sandbox folder**, and any path that leads outside it is refused.

### 2a. The stub and the sandbox (free)

In [ ]:
text_editor_tool = {"type": "text_editor_20250728", "name": "str_replace_based_edit_tool"}

SANDBOX = (OUTPUT_DIR / "text_editor_sandbox").resolve()   # the ONLY folder Claude may touch

SAMPLE_MAIN = """\
def greet(name):
    return f"Hello, {name}!"


def add(a, b):
    return a + b


if __name__ == "__main__":
    print(greet("world"))
    print(add(2, 3))
"""


def reset_sandbox():
    """Put the sandbox back to its starting state: just main.py."""
    SANDBOX.mkdir(parents=True, exist_ok=True)
    for file in SANDBOX.iterdir():
        if file.is_file():
            file.unlink()
    (SANDBOX / "main.py").write_text(SAMPLE_MAIN)


reset_sandbox()
print("Stub sent to Claude:", text_editor_tool)
print("Files in the sandbox:", sorted(p.name for p in SANDBOX.iterdir()))

# Even a stub isn't free: the tool's built-in description is added to every request.
question = [{"role": "user", "content": "Summarize ./main.py"}]
without_tool = client.messages.count_tokens(model=MODEL, messages=question).input_tokens
with_tool = client.messages.count_tokens(model=MODEL, messages=question, tools=[text_editor_tool]).input_tokens
print(f"\nThe text editor adds {with_tool - without_tool:,} input tokens to every request.")

### 2b. Your side of the tool (free)

One small function per command, plus a dispatcher. Errors are **raised**, and the tool loop sends them back to Claude as `is_error` results, as in notebook 05.

`safe_path()` is the important one: it turns the path Claude gave into a real path, and refuses it if it lands outside the sandbox. `resolve()` collapses tricks like `..` and follows links.

In [ ]:
def safe_path(path):
    """Turn a model-supplied path into a real path INSIDE the sandbox, or refuse."""
    resolved = (SANDBOX / path).resolve()
    if not resolved.is_relative_to(SANDBOX):
        raise PermissionError(f"Path '{path}' is outside the sandbox. Use paths like './main.py'.")
    return resolved


def editor_view(path, view_range=None):
    target = safe_path(path)
    if target.is_dir():
        return "\n".join(sorted(f.name + ("/" if f.is_dir() else "") for f in target.iterdir()))
    lines = target.read_text().splitlines()   # raises FileNotFoundError if the file is missing
    start, end = 1, len(lines)
    if view_range:
        start = view_range[0]
        end = len(lines) if view_range[1] == -1 else view_range[1]   # -1 means "to the end"
    return "\n".join(f"{number:>4}\t{lines[number - 1]}" for number in range(start, min(end, len(lines)) + 1))


def editor_create(path, file_text):
    target = safe_path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(file_text)
    return f"Created {path} ({len(file_text.splitlines())} lines)."


def editor_str_replace(path, old_str, new_str=""):
    target = safe_path(path)
    text = target.read_text()
    matches = text.count(old_str)
    if matches == 0:
        raise ValueError("old_str was not found in the file. View the file and copy the text exactly.")
    if matches > 1:
        raise ValueError(f"old_str appears {matches} times. Include more surrounding text so it matches exactly once.")
    target.write_text(text.replace(old_str, new_str))
    return f"Replaced 1 occurrence in {path}."


def editor_insert(path, insert_line, insert_text):
    target = safe_path(path)
    lines = target.read_text().splitlines()
    if not 0 <= insert_line <= len(lines):
        raise ValueError(f"insert_line must be between 0 and {len(lines)}.")
    lines[insert_line:insert_line] = insert_text.splitlines()
    target.write_text("\n".join(lines) + "\n")
    return f"Inserted text after line {insert_line} in {path}."


def run_text_editor(tool_input):
    """Route one text editor request to the matching function."""
    args = dict(tool_input)
    command = args.pop("command")
    if command == "view":
        return editor_view(**args)
    if command == "create":
        return editor_create(**args)
    if command == "str_replace":
        return editor_str_replace(**args)
    if command == "insert":
        return editor_insert(**args)
    raise ValueError(f"Unsupported command: {command}")


# Test it with no API call: we play Claude and send the same requests it would.
reset_sandbox()
print(run_text_editor({"command": "view", "path": "./main.py", "view_range": [1, 2]}))
print(run_text_editor({"command": "create", "path": "./notes.py", "file_text": "x = 1\ny = 2\n"}))
print(run_text_editor({"command": "str_replace", "path": "./notes.py", "old_str": "x = 1", "new_str": "x = 100"}))
print(run_text_editor({"command": "insert", "path": "./notes.py", "insert_line": 0, "insert_text": "# my notes"}))
print(run_text_editor({"command": "view", "path": "./notes.py"}))

assert (SANDBOX / "notes.py").read_text() == "# my notes\nx = 100\ny = 2\n"
reset_sandbox()

### 2c. Claude reads a file (paid)

This is the tool loop from notebook 05. Only `run_tool` is new: it sends every request to the text editor.

Claude can't see the file until it asks to **view** it. Reading before acting is how an agent finds out about its surroundings, and notebook 11 comes back to this idea.

In [ ]:
def run_tool(tool_name, tool_input):
    if tool_name == text_editor_tool["name"]:
        return run_text_editor(tool_input)
    raise ValueError(f"Unknown tool: {tool_name}")


def run_tools(response):
    """Run every tool_use block in a response, and return the matching tool_result blocks."""
    results = []
    for block in response.content:
        if block.type != "tool_use":
            continue
        try:
            output = run_tool(block.name, block.input)
            results.append({"type": "tool_result", "tool_use_id": block.id, "content": str(output)})
        except Exception as error:
            results.append({"type": "tool_result", "tool_use_id": block.id,
                            "content": f"{type(error).__name__}: {error}", "is_error": True})
        command = block.input.get("command")
        print(f"      {command} {block.input.get('path')} -> {results[-1]['content'][:70]!r}")
    return results


def run_conversation(messages, tools, max_turns=10, max_tokens=4000):
    """Call Claude, run any tools it asks for, and repeat until it answers."""
    for turn in range(1, max_turns + 1):
        response = track(client.messages.create(
            model=MODEL,
            max_tokens=max_tokens,
            output_config={"effort": "low"},
            messages=messages,
            tools=tools,
        ))
        add_assistant_message(messages, response)
        print(f"Turn {turn}: stop_reason={response.stop_reason}")
        if response.stop_reason != "tool_use":
            print("\nFinal answer:", text_from_message(response))
            return response
        add_user_message(messages, run_tools(response))
    raise RuntimeError(f"Stopped after {max_turns} turns without a final answer.")


reset_sandbox()
messages = []
add_user_message(messages, "Open the ./main.py file and summarize what it does in 2 sentences.")
final = run_conversation(messages, [text_editor_tool])

assert final.stop_reason == "end_turn"

### 2d. Claude edits a file and writes tests (paid)

Now Claude changes the file and creates a new one. Watch which commands it picks.

Afterwards we **run the tests Claude wrote**, to check its work. That executes code a model wrote, on your machine, so we show you the files first. The sandbox folder limits where Claude can **write**, but it is not a security sandbox for **running** code: the tests run with your own permissions. Set `RUN_CLAUDES_TESTS = False` if you'd rather read them than run them.

In [ ]:
RUN_CLAUDES_TESTS = True

reset_sandbox()
messages = []
add_user_message(messages, (
    "Open the ./main.py file and add a function called calculate_pi that returns pi rounded to 5 decimal places. "
    "Then create a ./test_main.py file that tests it using Python's built-in unittest module."
))
run_conversation(messages, [text_editor_tool], max_turns=12)

print("\nFiles in the sandbox:", sorted(p.name for p in SANDBOX.iterdir()))
for name in ["main.py", "test_main.py"]:
    if (SANDBOX / name).exists():
        print(f"\n----- {name} -----")
        print((SANDBOX / name).read_text())

assert (SANDBOX / "test_main.py").exists(), "Claude didn't create test_main.py. Re-run this cell."

if RUN_CLAUDES_TESTS:
    result = subprocess.run([sys.executable, "-m", "unittest", "test_main"], cwd=SANDBOX,
                            capture_output=True, text=True, timeout=60)
    print((result.stdout + result.stderr).strip())
    print("Tests", "PASSED" if result.returncode == 0 else "FAILED")

**What to notice**

- Claude probably **viewed** `main.py` before changing it. To use `str_replace`, it needs the exact current text.
- It likely used `str_replace` or `insert` for the edit, and `create` for the new file. Your code did every write.
- The tests check Claude's own work. Code that has been run is far more trustworthy than code that only looks right.

### 2e. When a request is unsafe or wrong

First, with no API call: each of these must be refused.

In [ ]:
reset_sandbox()

bad_requests = {
    "path escapes with ..": {"command": "view", "path": "../../06_built_in_tools.ipynb"},
    "absolute path": {"command": "view", "path": "/etc/hosts"},
    "old_str not found": {"command": "str_replace", "path": "./main.py", "old_str": "nope", "new_str": "x"},
    "old_str matches twice": {"command": "str_replace", "path": "./main.py", "old_str": "return", "new_str": "x"},
    "removed command": {"command": "undo_edit", "path": "./main.py"},
}

for label, request in bad_requests.items():
    try:
        run_text_editor(request)
        raise AssertionError(f"{label} was NOT refused")
    except (PermissionError, ValueError) as error:
        print(f"Refused, {label}: {type(error).__name__}: {error}")

Now through Claude (paid). The refusal goes back as an `is_error` result, and Claude explains it instead of crashing.

In [ ]:
messages = []
add_user_message(messages, "Use the text editor to view the file ../outside.txt and tell me what it says.")
final = run_conversation(messages, [text_editor_tool], max_tokens=1000)

assert not (SANDBOX.parent / "outside.txt").exists()

## 3. Web search: a tool Anthropic runs (paid)

Web search is a **server tool**. In **one** API call, Claude decides to search, Anthropic's servers run the search, and Claude reads the results and answers. There's no `run_tool`, no `tool_result` and no loop. Your code only reads the response.

**Before you run this:** web search is on for every organization unless an administrator has turned it off, at [Settings > Capabilities](https://platform.claude.com/settings/capabilities) in the Console. If it's off, this request fails with a 400 error saying web search is not enabled.

**Versions.** `web_search_20260209` adds **dynamic filtering**: Claude writes and runs code that filters the search results before they reach its context, which can save input tokens. It runs inside code execution that the API sets up by itself, at no extra charge, so **don't add the code execution tool alongside it**. A newer `web_search_20260318` also lets you leave raw results out of the response. The basic `web_search_20250305` has no filtering.

**Cost:** $10 per 1,000 searches, plus the results, which Claude reads as **input tokens**. A search that fails isn't billed. `max_uses` caps the number of searches in one request.

The response mixes several block types:

| Block `type` | What it holds |
|---|---|
| `server_tool_use` | What Claude ran: a search query, or code for dynamic filtering |
| `web_search_tool_result` | A list of results (`url`, `title`, `page_age`), or an **error object** |
| `text` | Claude's answer, split into many blocks. Some carry `citations` |

We restrict results to one trusted domain with `allowed_domains`, and allow at most 2 searches.

In [ ]:
web_search_tool = {
    "type": "web_search_20260209",
    "name": "web_search",
    "max_uses": 2,                      # at most 2 searches in this request
    "allowed_domains": ["nih.gov"],     # also matches subdomains, such as pubmed.ncbi.nlm.nih.gov
}


def ask_with_search(question, tool, max_tokens=3000):
    """Send one question with web search enabled. Return the last response and every block of the turn."""
    messages = [{"role": "user", "content": question}]
    blocks = []
    for _ in range(3):
        response = track(client.messages.create(
            model=MODEL,
            max_tokens=max_tokens,
            output_config={"effort": "low"},
            messages=messages,
            tools=[tool],
        ))
        blocks += response.content
        if response.stop_reason != "pause_turn":
            return response, blocks
        # pause_turn: the server paused a long turn. Send everything so far back unchanged,
        # and Claude carries on. The next response holds only the NEW blocks, so we keep them all.
        print("(pause_turn, continuing the same turn)")
        messages = [messages[0], {"role": "assistant", "content": blocks}]
    return response, blocks


search_response, search_blocks = ask_with_search(
    "How many minutes of moderate exercise per week do adults need for health benefits? Answer in 3 sentences.",
    web_search_tool,
)

print("\nstop_reason:", search_response.stop_reason)
print("Blocks in the turn, in order:")
for number, block in enumerate(search_blocks):
    print(f"  {number:>2}. {block.type}")

Now take the response apart: what Claude searched for, what came back, and what it cited.

In [ ]:
from urllib.parse import urlparse

queries, result_domains, citations = [], set(), []

for block in search_blocks:
    if block.type == "server_tool_use":
        queries.append(f"{block.name}: {str(block.input)[:100]}")
    elif block.type == "web_search_tool_result":
        if not isinstance(block.content, list):       # an error object, not a list of results
            print("Search error:", block.content.error_code)
            continue
        for result in block.content:
            result_domains.add(urlparse(result.url).netloc)
    elif block.type == "text" and block.citations:
        citations.extend(block.citations)

print("What Claude ran:")
for query in queries:
    print("  ", query)
print("\nDomains in the results:", sorted(result_domains))
# The answer is the text AFTER the last search step. Earlier text is Claude narrating its plan.
last_tool_step = max((i for i, block in enumerate(search_blocks) if block.type != "text"), default=-1)
print(f"\nAnswer:\n{full_text(search_blocks[last_tool_step + 1:])}")
print(f"\n{len(citations)} citations. The first few:")
for citation in citations[:3]:
    print(f"   {citation.title[:60]}  {citation.url}")
    print(f"      \"{' '.join(citation.cited_text.split())[:150]}\"")

if result_domains:
    assert all(domain == "nih.gov" or domain.endswith(".nih.gov") for domain in result_domains)
else:
    print("\nNo search results came back. Claude may have answered from memory, so re-run section 3.")

**What to notice**

- **Look at `searches=` and `input=` on the cost line.** The search results count as input tokens, so one answer can cost more than you expect.
- Every domain in the results is `nih.gov` or one of its subdomains. That's `allowed_domains` at work. Use `blocked_domains` for the opposite, but never both in one tool.
- Each citation gives the exact quoted text and its URL. **If you show these answers to users, show the citations too.**
- Errors don't raise an exception. A failed search comes back as a normal 200 response, where `content` is an error object such as `max_uses_exceeded` instead of a list. That's why the code checks `isinstance(block.content, list)`.
- **To ask a follow-up question,** append the whole `response.content` to the history. The results carry `encrypted_content`, which the API needs to restore them, and a changed or missing one gives a 400 error.

## 4. Code execution and the Files API (paid)

Claude is good at deciding **what** to compute, but arithmetic done in its head can slip, as the meal plans in notebook 04 showed. Code execution gives Claude a Python sandbox **on Anthropic's servers**: it writes code, runs it, reads the output, and fixes its own mistakes, all inside one API call.

Two features work together:

- **Files API:** `client.files.upload(...)` stores a file once and gives you a `file_id` to use in later requests.
- **Code execution:** `{"type": "code_execution_20260521", "name": "code_execution"}`, a server tool. A `{"type": "container_upload", "file_id": ...}` block puts your uploaded file inside the sandbox.

**How files come back:** each command Claude runs gets a fresh, empty folder, `$OUTPUT_DIR`. Files saved there are returned as new `file_id`s that you can download. The sandbox has no internet access, and comes with pandas, matplotlib and other common libraries.

**Cost:** tokens as usual. Execution time is billed separately, at $0.05 per container hour, after **1,550 free hours** per organization each month, with a 5-minute minimum per run. A notebook like this stays well inside the free hours, so `track()` doesn't count it. Code execution is free when the request also includes web search `20260209` or later.

### 4a. Upload the data (free)

A small table of Halvora's KPIs, taken from `data/halvora_report.md`. The report doesn't state 2022 revenue, so we work it out: $412M after 14% growth means 412 / 1.14, which is about $361.4M.

In [ ]:
kpi_csv = """division,metric,2022,2023
Company,Revenue ($M),361.4,412
Company,Operating margin (%),9.1,11.4
Engineering,Bugs fixed,1037,1284
Engineering,Bugs found before release (%),61,74
Engineering,Customer-facing outages,19,7
Engineering,Average recovery time (min),52,18
HR,Employees,1420,1730
"""

uploaded = client.files.upload(file=("halvora_kpis.csv", io.BytesIO(kpi_csv.encode("utf-8")), "text/csv"))
print(f"Uploaded {uploaded.filename}: file_id={uploaded.id}, {uploaded.size_bytes} bytes")

files_to_delete = [uploaded.id]   # we clean up everything we create, in 4c

### 4b. Let Claude analyse it (paid)

One request. Claude explores the file, writes the analysis, runs it, and saves a chart. We print every step it took.

A long run can stop with `pause_turn`. To continue, send what you have so far back **with its container ID**, so Claude keeps the files it already made. Each continuation returns only the new blocks, so we collect them all in `blocks`.

In [ ]:
code_execution_tool = {"type": "code_execution_20260521", "name": "code_execution"}

messages = [{"role": "user", "content": [
    {"type": "text", "text": "Load the CSV. Compute the % change from 2022 to 2023 for every metric and print it "
                             "as a table. Save a horizontal bar chart of the % changes as kpi_changes.png in "
                             "$OUTPUT_DIR. Finish with a 2-sentence summary for the board."},
    {"type": "container_upload", "file_id": uploaded.id},
]}]

container_id = None
blocks = []   # every block of the turn, across any pause_turn continuations
for _ in range(3):
    params = {"model": MODEL, "max_tokens": 8000, "output_config": {"effort": "low"},
              "messages": messages, "tools": [code_execution_tool]}
    if container_id:
        params["container"] = container_id   # keep working in the same sandbox
    response = track(client.messages.create(**params))
    container_id = response.container.id if response.container else container_id
    blocks += response.content
    if response.stop_reason != "pause_turn":
        break
    print("(pause_turn, continuing in the same container)")
    messages = [messages[0], {"role": "assistant", "content": blocks}]

output_file_ids = []
for block in blocks:
    if block.type == "server_tool_use":
        detail = block.input.get("command") or block.input.get("code") or ""
        if block.input.get("path"):
            detail = f"{detail} {block.input['path']}"
        first_line = str(detail).strip().splitlines()[0] if str(detail).strip() else ""
        print(f"> {block.name}: {first_line[:90]}")
    elif block.type == "bash_code_execution_tool_result":
        result = block.content
        if result.type != "bash_code_execution_result":    # an error object
            print(f"   error: {result.error_code}")
            continue
        print(f"   return_code={result.return_code}")
        for line in (result.stdout or "").strip().splitlines()[:12]:
            print(f"   | {line}")
        if result.stderr:
            print(f"   | stderr: {result.stderr.strip()[:150]}")
        for item in result.content or []:
            output_file_ids.append(item.file_id)
            print(f"   output file: {item.file_id}")
    elif block.type == "text" and block.text.strip():
        print(f"\nClaude: {block.text.strip()}\n")

files_to_delete += output_file_ids
print("stop_reason:", response.stop_reason)

**What to notice**

- **One API call, several steps.** All of them ran on Anthropic's servers, which is what makes it a server tool.
- The numbers in the table come from **running code**, not from Claude's head.
- **Compare `input=` with earlier cells.** Every step's code and output is part of the request, so code execution calls are bigger than plain chat.
- Claude may save the chart more than once if it fixes a mistake, so the same filename can come back twice. 4c keeps the last one.

### 4c. Download the chart, then clean up (free)

Uploaded and generated files **stay in your account until you delete them**, so we delete everything this notebook created. Run this cell even if 4b failed.

`os.path.basename` strips any folder from the filename before we write it, so a file can't land outside our output folder.

In [ ]:
from IPython.display import Image, display

latest_by_name = {}
for file_id in output_file_ids:
    metadata = client.files.retrieve_metadata(file_id)
    latest_by_name[os.path.basename(metadata.filename)] = file_id

for filename, file_id in latest_by_name.items():
    local_path = OUTPUT_DIR / filename
    client.files.download(file_id).write_to_file(local_path)
    print(f"Downloaded {filename} ({local_path.stat().st_size:,} bytes)")
    if local_path.suffix == ".png":
        display(Image(str(local_path)))

for file_id in files_to_delete:
    client.files.delete(file_id)
print(f"Deleted {len(files_to_delete)} file(s) from the Files API.")
files_to_delete = []

## Final spend

In [ ]:
print(f"This notebook spent ${spent:.4f} of its ${BUDGET_USD:.2f} budget.")

## Takeaways

1. **There are three kinds of tools:** your own, Anthropic-defined ones you run, and server tools Anthropic runs.
2. **The text editor is a stub plus your code.** Claude sends commands, and your code does every read and write.
3. **Treat model-supplied paths as untrusted.** Resolve them and confine them to one folder.
4. **Web search is one call with no loop.** Results count as input tokens, plus $10 per 1,000 searches.
5. **Errors from server tools arrive inside a normal response.** Check for an error object, not just exceptions.
6. **Show citations** with any answer built from web results.
7. **Code execution plus the Files API** turns "probably right" arithmetic into computed results.
8. **Delete files you upload or generate.** They stay in your account until you do.

## Your turn

Each exercise costs a few cents or less.

- In 2d, ask Claude to *"add type hints and docstrings to ./main.py"*. Does it use `str_replace` or `create`?
- Make `editor_create` keep a `.bak` copy when it overwrites a file, as the documentation recommends, then ask Claude to rewrite `main.py`.
- In section 3, swap `allowed_domains` for `"blocked_domains": ["wikipedia.org"]` and ask a different question.
- In section 3, ask a question that needs no search, such as *"What is 2 + 2?"*. Does Claude still search?
- In 4b, ask for a second output file, such as an Excel sheet of the results.

**Next:** [07 · RAG I](07_rag_basics.ipynb): splitting a document into chunks, turning text into embeddings, and searching by meaning.